# Sentiment directions fitted at ADJ, VRB, SUM, and END positions

This Colab notebook runs the complete sentiment-position comparison for **GPT-2 Small** and **Qwen3-0.6B Base**. It fits mean difference, logistic regression, and one-dimensional DAS directions at the adjective (ADJ), verb (VRB), second `movie` (SUM), and final `is` (END) prompt-token positions for every non-embedding residual boundary. The complete ToyMovieReview ADVERB panel selects DAS checkpoints and layers; the selected directions are then evaluated on ADVERB, held-out ToyMovieReview ADJ, and the model-specific SST directed pairs.

All direction artifacts, result tables, provenance, and figures are written directly to one timestamped Google Drive directory. The notebook calls the `sentiment_geometry` package; it does not reimplement activation extraction, fitting, patching, metrics, or artifact validation.

## Before running

1. Choose **Runtime → Change runtime type → T4 GPU** or a larger GPU.
2. Have a Hugging Face token with read access to the SST dataset repository ready. The notebook will request it through a hidden input prompt.
3. Leave `RESUME_RUN_ID = None` for a new timestamped run. After a disconnection, paste the previous directory name into `RESUME_RUN_ID` to reuse compatible direction checkpoints.
4. The full sweep fits 480 directions and performs causal evaluation for each one. It can take a long time, particularly for DAS.

## 1. User settings

In [ ]:
PROJECT_URL = "https://github.com/Adefioye/sentiment-manifold.git"
PROJECT_REVISION = None  # Optional commit or tag. Existing checkouts must already match it.

DRIVE_STORAGE_ROOT = "/content/drive/MyDrive/sentiment-geometry"
TIMEZONE_NAME = "America/Chicago"
RESUME_RUN_ID = None  # Example: "2026-09-05_18-42_CDT"

DEVICE = "cuda"
DTYPE = "auto"
BATCH_SIZE = 16
RUN_EXPERIMENT = True

## 2. Clone and install the project

An existing checkout is reused. If `PROJECT_REVISION` is set, the notebook verifies the checkout rather than silently changing it. The editable install includes notebook dependencies. The repository root is then added explicitly to `sys.path`; this project has no `src/` directory. The cell verifies every required third-party dependency, removes any stale package modules from the current runtime, explicitly imports `sentiment_geometry` from this checkout, and imports the complete package module tree as an early installation check.

In [ ]:
import importlib
import os
import pkgutil
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/sentiment-manifold")
if not (PROJECT_ROOT / ".git").is_dir():
    subprocess.run(["git", "clone", PROJECT_URL, str(PROJECT_ROOT)], check=True)
else:
    print(f"Reusing {PROJECT_ROOT}")

project_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
if PROJECT_REVISION is not None:
    expected_commit = subprocess.check_output(
        ["git", "rev-parse", PROJECT_REVISION], cwd=PROJECT_ROOT, text=True
    ).strip()
    if project_commit != expected_commit:
        raise RuntimeError(
            f"Existing checkout is {project_commit}, expected {expected_commit}. "
            "Use a fresh runtime or update PROJECT_REVISION."
        )

# Show installation output so dependency or editable-install failures are visible.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{PROJECT_ROOT}[notebooks]"],
    check=True,
)
os.chdir(PROJECT_ROOT)

required_dependencies = {
    "datasets": "datasets",
    "huggingface_hub": "huggingface-hub",
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "pandas": "pandas",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "torch": "torch",
    "tqdm": "tqdm",
    "transformers": "transformers",
    "yaml": "pyyaml",
}
dependency_failures = {}
for import_name, distribution_name in required_dependencies.items():
    try:
        importlib.import_module(import_name)
    except Exception as error:
        dependency_failures[distribution_name] = f"{type(error).__name__}: {error}"
if dependency_failures:
    raise ImportError(f"Required dependency imports failed: {dependency_failures}")

# The package lives at PROJECT_ROOT/sentiment_geometry, not PROJECT_ROOT/src.
project_root_string = str(PROJECT_ROOT.resolve())
if project_root_string in sys.path:
    sys.path.remove(project_root_string)
sys.path.insert(0, project_root_string)
for loaded_name in list(sys.modules):
    if loaded_name == "sentiment_geometry" or loaded_name.startswith("sentiment_geometry."):
        del sys.modules[loaded_name]
importlib.invalidate_caches()

import sentiment_geometry
expected_package_root = (PROJECT_ROOT / "sentiment_geometry").resolve()
imported_package_root = Path(sentiment_geometry.__file__).resolve().parent
if imported_package_root != expected_package_root:
    raise ImportError(
        f"Imported sentiment_geometry from {imported_package_root}, "
        f"expected {expected_package_root}. Restart the runtime and rerun from the top."
    )

module_names = sorted(
    module.name
    for module in pkgutil.walk_packages(
        sentiment_geometry.__path__, prefix="sentiment_geometry."
    )
    if module.name != "sentiment_geometry.__main__"
)
for module_name in module_names:
    importlib.import_module(module_name)

required_apis = {
    "sentiment_geometry.experiments": {
        "SentimentPositionExperimentConfig",
        "run_sentiment_position_comparison",
        "audit_direction_artifacts",
    },
    "sentiment_geometry.persistence": {
        "RunArtifactStore",
        "maybe_mount_google_drive",
        "prepare_timestamped_run",
    },
}
for module_name, api_names in required_apis.items():
    module = importlib.import_module(module_name)
    missing_apis = sorted(name for name in api_names if not hasattr(module, name))
    if missing_apis:
        raise ImportError(
            f"{module_name} is missing {missing_apis}. "
            "The checkout is older than this notebook; use a fresh runtime."
        )

print("Project commit:", project_commit)
print("Imported package from:", imported_package_root)
print(f"Verified {len(required_dependencies)} required dependencies.")
print(f"Successfully imported {len(module_names) + 1} sentiment_geometry modules.")

## 3. Mount Google Drive and prepare the run

A new run uses a readable minute-level name such as `2026-09-05_18-42_CDT`. The same minute cannot be reused accidentally. Resume requires the exact prior run ID.

In [ ]:
import torch

from sentiment_geometry.persistence import maybe_mount_google_drive, prepare_timestamped_run

if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Enable a GPU runtime before continuing.")

maybe_mount_google_drive(True)
if "RUN_LAYOUT" not in globals() or RESUME_RUN_ID is not None:
    RUN_LAYOUT = prepare_timestamped_run(
        DRIVE_STORAGE_ROOT,
        experiment_name="sentiment-position-comparison",
        timezone_name=TIMEZONE_NAME,
        resume_run_id=RESUME_RUN_ID,
    )

print("Run ID:             ", RUN_LAYOUT.run_id)
print("Run directory:      ", RUN_LAYOUT.root)
print("Result tables:      ", RUN_LAYOUT.results_dir)
print("Direction artifacts:", RUN_LAYOUT.directions_dir)
print("Figures:            ", RUN_LAYOUT.figures_dir)
print("GPU:                ", torch.cuda.get_device_name(0))

## 4. Read the Hugging Face secret

As in the preprocessing notebook, `HF_TOKEN` is requested with `getpass`, cached only in a notebook-owned runtime dictionary, and removed after the experiment. Its value is never printed or written to Drive.

In [ ]:
from getpass import getpass

from huggingface_hub import HfApi

_RUNTIME_SECRETS = {}

def get_runtime_secret(name):
    if name not in _RUNTIME_SECRETS:
        value = getpass(f"Enter {name} (input hidden): ").strip()
        if not value:
            raise RuntimeError(f"{name} was not provided.")
        _RUNTIME_SECRETS[name] = value
    return _RUNTIME_SECRETS[name]

def delete_runtime_secret(name):
    value = _RUNTIME_SECRETS.pop(name, None)
    if value is not None:
        del value

_token = get_runtime_secret("HF_TOKEN")
hf_account = HfApi(token=_token).whoami()["name"]
del _token
print(f"Authenticated to Hugging Face as {hf_account}. Token value was not displayed.")

## 5. Load and lock the complete experiment configuration

This cell fixes both models, three fitting methods, four fitting positions, every boundary from `1..n_layers`, and all-token patching. The full ADVERB panel selects DAS epochs and layers; the frozen layer is then evaluated on ADVERB, ADJ, and SST. Boundary 0 is excluded because it is the embedding residual.

In [ ]:
from dataclasses import asdict

import pandas as pd
import yaml
from IPython.display import display

from sentiment_geometry.experiments import SentimentPositionExperimentConfig

CONFIG_PATH = PROJECT_ROOT / "configs/sentiment_position_comparison.yaml"
config = SentimentPositionExperimentConfig.load(CONFIG_PATH)

expected_models = ["gpt2-small", "qwen-0.6b"]
if [model.name for model in config.models] != expected_models:
    raise RuntimeError(f"Expected exactly {expected_models}; got {[m.name for m in config.models]}")

for model in config.models:
    model.device = DEVICE
    model.dtype = DTYPE
    model.batch_size = BATCH_SIZE

config.sweep.layers = "all_non_embedding"
config.sweep.methods = ["mean_diff", "logistic_regression", "das"]
config.sweep.fit_positions = ["adjective", "verb", "summary", "final"]
config.sweep.output_dir = str(RUN_LAYOUT.results_dir)
config.sweep.checkpoint_dir = str(RUN_LAYOUT.directions_dir)
config.sweep.resume = True
config.data.sst_max_directed_cases = None
expected_toy_evaluations = ["toy_adjectives", "toy_adverbs"]
expected_final_evaluations = ["toy_adverbs", "toy_adjectives", "sst"]
if config.data.toy_evaluations != expected_toy_evaluations:
    raise RuntimeError(
        f"Expected Toy evaluations {expected_toy_evaluations}; "
        f"got {config.data.toy_evaluations}."
    )
if config.selection.final_evaluations != expected_final_evaluations:
    raise RuntimeError(
        f"Expected final evaluations {expected_final_evaluations}; "
        f"got {config.selection.final_evaluations}."
    )
config.validate()

requested_config_path = RUN_LAYOUT.root / "requested_config.yaml"
requested_config_path.write_text(
    yaml.safe_dump(config.to_dict(), sort_keys=False), encoding="utf-8"
)
RUN_LAYOUT.update_manifest(
    status="configured",
    metadata={
        "models": [asdict(model) for model in config.models],
        "methods": config.sweep.methods,
        "fit_positions": config.sweep.fit_positions,
        "layers": config.sweep.layers,
        "evaluations": config.selection.final_evaluations,
        "configuration": requested_config_path.name,
    },
)

display(
    pd.DataFrame(
        [
            {
                "model": model.name,
                "hub_model": model.hub_name,
                "revision": model.revision,
                "device": model.device,
                "dtype": model.dtype,
                "batch_size": model.batch_size,
            }
            for model in config.models
        ]
    )
)
print("Methods:", config.sweep.methods)
print("Fitting positions:", config.sweep.fit_positions)
print("Layers: 1..n_layers (embedding boundary 0 excluded)")
print("Final evaluations:", config.selection.final_evaluations)

## 6. Record the software environment

In [ ]:
import platform
from importlib.metadata import version

from sentiment_geometry.persistence import RunArtifactStore

environment = {
    "project_commit": project_commit,
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": version("torch"),
    "transformers": version("transformers"),
    "datasets": version("datasets"),
    "numpy": version("numpy"),
    "pandas": version("pandas"),
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
RunArtifactStore(RUN_LAYOUT.root).write_json("environment.json", environment)
display(pd.Series(environment, name="value").to_frame())

## 7. Run both models

The package processes GPT-2 Small first, releases its model memory, and then processes Qwen3-0.6B Base. Every fitted direction is saved under `directions/` immediately. The CSV tables under `results/` are refreshed after every layer.

In [ ]:
from sentiment_geometry.experiments import run_sentiment_position_comparison

if not RUN_EXPERIMENT:
    print("Experiment execution is disabled. Set RUN_EXPERIMENT = True in the settings cell.")
else:
    previous_hf_token = os.environ.get(config.data.hf_token_env)
    os.environ[config.data.hf_token_env] = get_runtime_secret("HF_TOKEN")
    RUN_LAYOUT.update_manifest(status="running")
    try:
        completed_results_dir = run_sentiment_position_comparison(config)
        if completed_results_dir.resolve() != RUN_LAYOUT.results_dir.resolve():
            raise RuntimeError(f"Unexpected results directory: {completed_results_dir}")
        RUN_LAYOUT.update_manifest(status="experiment-completed")
    except BaseException as error:
        RUN_LAYOUT.update_manifest(
            status="failed", metadata={"failure_type": type(error).__name__}
        )
        raise
    finally:
        if previous_hf_token is None:
            os.environ.pop(config.data.hf_token_env, None)
        else:
            os.environ[config.data.hf_token_env] = previous_hf_token
        delete_runtime_secret("HF_TOKEN")

    print("Experiment results saved to:", completed_results_dir)

## 8. Audit all saved sentiment directions

The audit reconstructs the expected method × position × layer grid from each model's resolved configuration, checks the metadata for omissions or duplicates, and verifies that every referenced `.npz` file exists on Drive. With four fitting positions, the pinned architectures should produce 144 GPT-2 directions and 336 Qwen directions.

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

from sentiment_geometry.experiments import audit_direction_artifacts

try:
    direction_audit = audit_direction_artifacts(RUN_LAYOUT.results_dir)
    display(direction_audit.summary)
    direction_audit.require_complete()
    if direction_audit.expected_total != 480:
        raise RuntimeError(
            f"Expected 480 direction artifacts for the pinned architectures; "
            f"the resolved run expected {direction_audit.expected_total}."
        )
except BaseException as error:
    RUN_LAYOUT.update_manifest(
        status="audit-failed", metadata={"audit_failure_type": type(error).__name__}
    )
    raise
else:
    completed_at = datetime.now(ZoneInfo(RUN_LAYOUT.timezone_name)).isoformat(timespec="minutes")
    RUN_LAYOUT.update_manifest(
        status="completed",
        metadata={
            "completed_at": completed_at,
            "direction_artifacts": direction_audit.expected_total,
            "direction_audit_complete": True,
        },
    )
    print(f"Verified all {direction_audit.expected_total} direction artifacts on Drive.")

## 9. Load the saved CSV tables

All analysis below begins by reopening the saved tables from Drive.

In [ ]:
MODEL_NAMES = ["gpt2-small", "qwen-0.6b"]

def load_saved_table(filename):
    tables = []
    for model_name in MODEL_NAMES:
        path = RUN_LAYOUT.results_dir / model_name / filename
        if not path.is_file():
            raise FileNotFoundError(path)
        table = pd.read_csv(path)
        if "model" not in table.columns:
            table.insert(0, "model", model_name)
        tables.append(table)
    return pd.concat(tables, ignore_index=True)

dataset_summary = load_saved_table("dataset_summary.csv")
metrics = load_saved_table("metrics.csv")
layer_selection = load_saved_table("layer_selection.csv")
selected_metrics = load_saved_table("selected_metrics.csv")
direction_similarities = load_saved_table("direction_similarities.csv")
direction_metadata = load_saved_table("direction_metadata.csv")

display(dataset_summary)
print(f"Loaded {len(metrics):,} aggregate metric rows from Drive.")
print(f"Loaded {len(direction_metadata):,} direction metadata rows from Drive.")

## 10. ADVERB-selected layers and frozen-layer results

Each model × method × fitting-position direction selects one boundary using ADVERB logit-flip percent. ADVERB, ADJ, and SST metrics are then recomputed and reported at that same frozen boundary.

In [ ]:
display(layer_selection.sort_values(["model", "fit_position", "method"]).reset_index(drop=True))
display(selected_metrics.sort_values(["model", "fit_position", "method", "dataset"]).reset_index(drop=True))

## 11. Direction similarity at first, middle, and last boundaries

The first table compares fitting methods within each activation position. The second retains the original adjective-versus-final endpoint comparison for each method. Absolute cosine is shown as the primary similarity; signed cosine remains available for orientation auditing.

In [ ]:
within_position = direction_similarities[
    (direction_similarities["position_a"] == direction_similarities["position_b"])
    & (direction_similarities["method_a"] < direction_similarities["method_b"])
][
    ["model", "layer", "position_a", "method_a", "method_b", "absolute_cosine", "signed_cosine"]
].rename(columns={"position_a": "fit_position"})

between_positions = direction_similarities[
    (direction_similarities["position_a"] == "adjective")
    & (direction_similarities["position_b"] == "final")
    & (direction_similarities["method_a"] == direction_similarities["method_b"])
][
    ["model", "layer", "method_a", "absolute_cosine", "signed_cosine"]
].rename(columns={"method_a": "method"})

print("Method-to-method similarity within each fitting position")
display(within_position.sort_values(["model", "layer", "fit_position", "method_a", "method_b"]))
print("Adjective-to-final similarity for each fitting method")
display(between_positions.sort_values(["model", "layer", "method"]))

## 12. Plot all causal metrics across fitting positions

Each figure uses only `selected_metrics.csv`. Within every fitting-method × evaluation-dataset panel, the line connects the four categorical activation positions ADJ, VRB, SUM, and END. Each point is evaluated at the layer selected for that particular model × method × fitting-position direction using ADVERB logit-flip percent; the connecting line is therefore a visual comparison across positions, not a layer trajectory.

In [ ]:
from itertools import product

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
POSITION_ORDER = ["adjective", "verb", "summary", "final"]
POSITION_LABELS = ["ADJ", "VRB", "SUM", "END"]
METHOD_ORDER = ["mean_diff", "logistic_regression", "das"]
DATASET_ORDER = ["toy_adverbs", "toy_adjectives", "sst"]
metric_specs = {
    "logit_difference_percent": "Logit difference (%)",
    "logit_flip_percent": "Logit flip (%)",
    "sign_flip_percent": "Literal sign flip (%)",
}
required_plot_columns = {
    "model", "method", "fit_position", "dataset", "selected_layer", *metric_specs
}
missing_columns = sorted(required_plot_columns - set(selected_metrics.columns))
if missing_columns:
    raise RuntimeError(f"selected_metrics.csv is missing columns: {missing_columns}")
if selected_metrics.duplicated(["model", "method", "fit_position", "dataset"]).any():
    raise RuntimeError("selected_metrics.csv contains duplicate result cells.")
expected_cells = set(product(MODEL_NAMES, METHOD_ORDER, POSITION_ORDER, DATASET_ORDER))
actual_cells = set(
    selected_metrics[["model", "method", "fit_position", "dataset"]]
    .itertuples(index=False, name=None)
)
missing_cells = sorted(expected_cells - actual_cells)
if missing_cells:
    raise RuntimeError(f"Selected-layer evaluations are incomplete: {missing_cells}")

position_metrics = selected_metrics.copy()
position_metrics["fit_position"] = pd.Categorical(
    position_metrics["fit_position"], categories=POSITION_ORDER, ordered=True
)
for model_name in MODEL_NAMES:
    model_metrics = position_metrics[position_metrics["model"] == model_name]
    model_figure_dir = RUN_LAYOUT.figures_dir / model_name
    model_figure_dir.mkdir(parents=True, exist_ok=True)
    for metric_column, y_label in metric_specs.items():
        grid = sns.relplot(
            data=model_metrics,
            x="fit_position",
            y=metric_column,
            row="method",
            col="dataset",
            row_order=METHOD_ORDER,
            col_order=DATASET_ORDER,
            kind="line",
            marker="o",
            sort=False,
            color="#0072B2",
            facet_kws={"sharey": True, "margin_titles": True},
            height=2.8,
            aspect=1.2,
        )
        grid.set_axis_labels("Fitting activation position", y_label)
        grid.set_xticklabels(POSITION_LABELS)
        grid.set_titles(row_template="{row_name}", col_template="{col_name}")
        grid.fig.suptitle(f"{model_name}: {y_label} across fitting positions", y=1.02)
        grid.fig.subplots_adjust(top=0.92)
        figure_path = model_figure_dir / f"{metric_column}_by_fit_position.png"
        grid.savefig(figure_path, dpi=180, bbox_inches="tight")
        plt.show()
        plt.close(grid.fig)
        print("Saved:", figure_path)

## 13. Replot the first/middle/last cosine matrices

In [ ]:
for model_name in MODEL_NAMES:
    model_similarity = direction_similarities[direction_similarities["model"] == model_name]
    model_figure_dir = RUN_LAYOUT.figures_dir / model_name
    for layer in sorted(model_similarity["layer"].unique()):
        layer_similarity = model_similarity[model_similarity["layer"] == layer]
        matrix = layer_similarity.pivot(
            index="direction_a", columns="direction_b", values="absolute_cosine"
        )
        figure, axis = plt.subplots(figsize=(8, 6))
        sns.heatmap(matrix, vmin=0, vmax=1, cmap="viridis", annot=True, fmt=".2f", ax=axis)
        axis.set_title(f"{model_name}: absolute cosine at boundary {layer}")
        axis.set_xlabel("Direction")
        axis.set_ylabel("Direction")
        figure.tight_layout()
        figure_path = model_figure_dir / f"absolute_cosine_boundary_{int(layer):02d}.png"
        figure.savefig(figure_path, dpi=180, bbox_inches="tight")
        plt.show()
        plt.close(figure)
        print("Saved:", figure_path)

## 14. Final Drive locations

In [ ]:
print("Completed run:", RUN_LAYOUT.run_id)
print("Manifest:     ", RUN_LAYOUT.manifest_path)
print("Configuration:", RUN_LAYOUT.root / "requested_config.yaml")
print("Environment:  ", RUN_LAYOUT.root / "environment.json")
print("Directions:   ", RUN_LAYOUT.directions_dir)
print("Results:      ", RUN_LAYOUT.results_dir)
print("Figures:      ", RUN_LAYOUT.figures_dir)

In [ ]:
_RUNTIME_SECRETS.clear()